In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt

from astropy import units as u
from astropy.modeling.powerlaws import BrokenPowerLaw1D

from prism.data import Spectrum
import prism.modeling.models as models
import prism.modeling.fitting as fitting
from prism.modeling.fitting import tie
import prism.modeling.display # importing the display module enables the model display features

plt.rcParams['axes.xmargin'] = 0

In [ ]:
filename = 'data/agnspec.txt'

# (Optional) Set the source coordinates and redshift
dec, ra = -25.12345, 13.12345
redshift = 0.0622

In [ ]:
spec = Spectrum.from_txt(filename=filename,
                         ra=ra,
                         dec=dec,
                         redshift=redshift,
                         xunit=u.AA,
                         yunit=1e-15 * u.erg / (u.s * u.cm**2 * u.AA),
                         name='AGN Spectrum')

# # (Example) Mask a region
# wmask = (spec.wave <= 5800) | (spec.wave >= 5980)
# spec = spec.cutout(mask=wmask)

# # (Example) Crop a portion of the spectrum
# wbounds = (4800., 7500.) # Å
# spec = spec.cutout(min=wbounds[0], max=wbounds[1])

from prism.utils.tools import get_ebv_from_map, apply_extinction_correction
# Apply Galactic extinction correction
ebv = get_ebv_from_map(ra, dec)
spec.y, spec.yerr = apply_extinction_correction(wave=spec.x, flux=spec.y, fluxerr=spec.yerr, ebv=ebv, rv=3.1)

# (Optional) Apply redshift correction to rest frame
spec.x /= (1 + spec.redshift)
spec.y *= (1 + spec.redshift)
spec.yerr *= (1 + spec.redshift)

spec.plot_spectrum() 

In [ ]:
# (Optional) Copy locally the CSV file containing the line list, and set up the local line database.
models.setup_local_lines(min=np.min(spec.x), max=np.max(spec.x), overwrite=True)

# # If you want to blindly proceed with the default line list without inspecting it, you can use models.set_wavelength_range()
# # This will just filter the lines to those that fall within the wavelength range of interest.
# models.set_wavelength_range(wmin=np.min(spec.x), wmax=np.max(spec.x))

# NOTE: If you decide to work in the observed frame, you should remember to set the line database to the rest frame, i.e. models.set_wavelength_range(wmin=np.min(spec.x)/(1+spec.redshift), wmax=np.max(spec.x)/(1+spec.redshift))

In [ ]:
# Line Models support instumental broadening. This can be set to either a constant value (in km/s) or a 2D array with wavelength and FWHM values (in km/s).
# The latter is useful when the instrumental resolution varies significantly across the wavelength range of interest.
# For the sake of simplicity we will use a constant value here, but I leave a snippet below to show how to set up a wavelength-dependent instrumental broadening based on the MUSE instrumental response.

instfwhm = 0. # km/s


# # Set wavelength-dependent instrumental broadening based on the MUSE instrumental response
# from astropy.constants import c
# c_kms = c.to('km/s').value

# # Instrumental Response of MUSE from the MUSE User Manual
# resp_lambda = np.array([4650.0, 5000.0, 5500.0, 6000.0, 6500.0, 7000.0, 7500.0, 8000.0, 8500.0, 9000.0, 9350.0])
# resp_R = np.array([1609, 1750, 1978, 2227, 2484, 2737, 2975, 3183, 3350, 3465, 3506])

# wave_rest = resp_lambda * (1.0 + redshift) # Null the zcorrection for instrumental effects. Not needed if you are working in the observed frame.
# fwhm_km_s = c_kms / resp_R

# instfwhm = np.array([wave_rest, fwhm_km_s]).T

In [ ]:
# Setup the model

# Continuum
bknpower = BrokenPowerLaw1D(
    name='bknpower',
    x_break=5450,
    amplitude=np.median(spec.y),
    alpha_1=2.0,
    alpha_2=2.0,
    bounds={
        'x_break': (5100, 5710),
        'amplitude': (1.5, 5.0),
        'alpha_1': (0.0, 3.0),
        'alpha_2': (0.0, 4.5),
    }
)

# Broad Line Region (BLR)
blr = models.agn.blr(
    name='blr',
    amplitude=10,
    offset=-360,
    fwhm=3727,
    instfwhm=instfwhm,
    bounds={
        'amplitude': (0, 1000),
        'offset': (-1000, 1000),
        'fwhm': (500, 8000),
    },
)

# Narrow Line Region (NLR)
nlr = models.agn.nlr(
    name='nlr',
    amplitude=1,
    offset=0,
    fwhm=500,
    instfwhm=instfwhm,
    bounds={
        'amplitude': (0, 100),
        'offset': (-300, 300),
        'fwhm': (0, 800),
    }
)

# Let's add Hydrogen and Helium lines tied to the NLR component. By default they are kept separate.
hhe_nlr = models.GaussianLines.from_csv(
    name='hhe_nlr',
    csv_files=['hydrogen.ecsv', 'helium.ecsv'],
    amplitude=1,
    offset=0,
    fwhm=500,
    instfwhm=instfwhm,
    bounds={
        'amplitude': (0, 100),
        'offset': (-300, 300),
        'fwhm': (0, 800),
    }
)
# Link the kinematics of the Hydrogen and Helium lines to those of the NLR component.
hhe_nlr.offset.tied = tie("nlr", lambda m: m.offset)
hhe_nlr.fwhm.tied = tie("nlr", lambda m: m.fwhm)

# OIII outflow
oiii_out = models.GaussianLines.from_csv(
    name='oiii',
    csv_files=['oiii.ecsv'],
    amplitude=1,
    offset=-560,
    fwhm=850,
    instfwhm=instfwhm,
    bounds={
        'amplitude': (0, 100),
        'offset': (-1000, 750),
        'fwhm': (50, 1500),
    }
)

# FeII Model
fe = models.agn.fe(
    name='fe',
    offset=0,
    fwhm=1200,
    instfwhm=instfwhm,
    bounds={
        'amplitude': (0, 100),
        'offset': (-3000, 3000),
        'fwhm': (800, 3000),
    }
)


# Combine all components
model = bknpower + blr + nlr + hhe_nlr + oiii_out + fe

In [ ]:
from prism.modeling import instrument

# Create the wavelength grid in the observer frame (not redshift corrected)
# Standard grid for agnspec.txt: 4700 to 7560 Angstrom
wave_obs = np.arange(4700, 7561.25, 1.25)

# Define the FWHM gradient from 100 to 120 km/s
fwhm_kms = np.linspace(100, 120, len(wave_obs))

# Convert FWHM from km/s to Angstrom: FWHM_ang = (v / c) * lambda
from astropy.constants import c
c_kms = c.to('km/s').value
fwhm_ang = (fwhm_kms / c_kms) * wave_obs

# Calculate Gaussian sigma values
sigmas = fwhm_ang / (2 * np.sqrt(2 * np.log(2)))

# Build the sparse response matrix
# Using the internal helper from InstrumentResponse
matrix = instrument.InstrumentResponse._build_sparse_gaussian_matrix(wave_obs, sigmas)

# Create the InstrumentResponse instance (temporary, not saved to disk)
custom_ir = instrument.InstrumentResponse(wave_obs, matrix)

# Apply to the model using SpectralResponse (passing the instance directly)
# This keeps it temporary for this session and does not write to the archive.
lsf = instrument.SpectralResponse(instrument=custom_ir, wave=spec.x, z=spec.redshift, name='lsf')
model = model | lsf # Apply the response

In [ ]:
model

In [ ]:
fitter = fitting.TRFLSQFitter()
fitter.max_evaluations = 5000

# fitter = fitting.SherpaLM() # It depends case by case, but very robust alternative for bounded problems to astropy defaults! Ensure to have Sherpa installed in your environment.

fitted_model = fitter(model=model, x=spec.x, y=spec.y, yerr=spec.yerr, inplace=True)

In [ ]:
# Compute chi-squared
residuals = spec.y - fitted_model(spec.x)
chi_squared = np.sum((residuals / spec.yerr) ** 2)

print(f"Chi-squared: {chi_squared:.2f}")

In [ ]:
# Manually print the best-fit parameters
for name in fitted_model.param_names:
    param = getattr(fitted_model, name)
    if hasattr(param, 'std') and param.std is not None:
        print(f"  {name:16s}: {param.value:8.4f} ± {param.std:2g}")
    else:
        print(f"  {name:16s}: {param.value:8.4f}")

In [ ]:
plt.figure(figsize=(8, 4))

comps = models.get_components(model, additive=True)
for name in comps.names:
    plt.plot(spec.x, comps[name](spec.x), label=name)

plt.plot(spec.x, spec.y, label='Data', color='black')
plt.plot(spec.x, model(spec.x), label='Model', color='red')
# plt.plot(spec.x[100:], model(spec.x[100:]), label='Model')
plt.xlabel(f'Rest Wavelength ({spec.xunit})')
plt.ylabel(f'Flux ({spec.yunit})')

# plt.ylim(bottom=0.1)
# plt.yscale('log')

plt.title(spec.name)

plt.show()

In [ ]:
# Our line models support the .flux attribute, which gives us the integrated flux of the line.
nlr.flux

In [ ]:
# If the fitter uses covariance in the fitting process, by default we will add the .std attribute to the parameters of the model.
# In some cases you might want to estimate the uncertainties on the best-fit parameters using more robust methods, such as bootstrapping.
# We implement it in an easy way in prism, and it can be used with any of the available fitters.
# We perturb the data around their uncertainties, and we refit the model to each perturbed dataset.
# This will give us a distribution of best-fit parameters from which we can estimate the uncertainties.
# We supprt parallelized fitting! For simple models and small datasets, you can set nproc=1 to avoid the overhead of parallelization.
# For more complex models and larger datasets, you can increase nproc to speed up the process.

from prism.modeling.fitting.uncertainty import bootstrap

fitter = fitting.TRFLSQFitter()
fitter.max_evaluations = 5000

# fitter = fitting.ScipyTRF()

# fitter = fitting.SherpaLM()

samples = bootstrap(model=fitted_model, 
                    fitter=fitter,
                    x=spec.x, y=spec.y, yerr=spec.yerr,
                    n_samples=200, seed=11,
                    nproc=4,
                    noise_dist='uniform'
                    )

In [ ]:
# Print the all the best-fit parameters
for name in fitted_model.param_names:
    param = getattr(fitted_model, name)
    if hasattr(param, 'lolim') and param.lolim is not None:
        print(f"  {name:16s}: [{param.lolim:8.4f}, {param.uplim:8.4f}]")
    else:
        print(f"  {name:16s}: {param.value:8.4f}")

In [ ]:
# Print the paramters with fluxes
components = models.get_components(fitted_model)

for i, name in enumerate(components.names):
    comp = components[i]

    if isinstance(comp, models.LineGroupBase):
        for line in comp.lines[comp.lines['weight'] == 1]['name']: # Drop lines with weight!=1. Repeated entries otherwise.
            flux = comp.flux.loc[line, 'value']
            flux_limits = comp.flux.loc[line, ['lolim', 'uplim']]
            if flux_limits.notnull().any():
                print(f"  {name:12s} flux_{line:15s} [{flux_limits['lolim']:8.4f}, {flux_limits['uplim']:8.4f}]")
            else:
                print(f"  {name:12s} flux_{line:15s} {flux:8.4f} ± {comp.flux.loc[line, 'std']:2g}")

    for parname in comp.param_names:
        param = getattr(comp, parname)
        if hasattr(param, 'lolim') and param.lolim is not None:
            print(f"  {name:12s} {parname:20s} [{param.lolim:8.4f}, {param.uplim:8.4f}]")
        else:
            print(f"  {name:12s} {parname:20s} {param.value:8.4f}")


In [ ]:
from prism.utils import fmt

# Print the all the best-fit parameters
for name in fitted_model.param_names:
    param = getattr(fitted_model, name)
    value = param.median if hasattr(param, 'median') else param.value
    lolim = getattr(param, 'lolim', None)
    uplim = getattr(param, 'uplim', None)
    print(f"  {name:16s}: {fmt(value=value, lo=lolim, hi=uplim)}")


In [ ]:
from prism.utils import tex

for name in fitted_model.param_names:
    param = getattr(fitted_model, name)
    value = param.median if hasattr(param, 'median') else param.value
    lolim = getattr(param, 'lolim', None)
    uplim = getattr(param, 'uplim', None)
    print(f"  {name:16s}: {tex(value=value, lo=lolim, hi=uplim, use_sci=True, maxexp=4)}")

In [ ]:
# Our line models support the .flux attribute, which gives us the integrated flux of the line.
nlr.flux

In [ ]:
fe.flux

In [ ]:
# Access specific entry (pandas.DataFrame)
fe.flux.loc['FeIIa4G', 'value']

In [ ]:
nlr.lines

In [ ]:
nlr.eqw() # Compute the Equivalent Width of the line. By default it uses the local continuum, but you can also specify a different continuum model if you want.

# Access the computed EW
nlr.ew

In [ ]:
fitted_model.show()

In [ ]:
# The user can also save the fitted model and its parameters to a file, and load them later for inspection or further analysis, without the need to re-run the fitting process from scratch.
from prism.modeling.io import load_params, load_model

outfile = "results/agnmodel.fits"
fitted_model.save(outfile, overwrite=True)

lp = load_params(outfile)
lm = load_model(outfile)
print("amp_hb:", lp["amp_hb4861_1"])
print("model class:", type(lm).__name__)